# Project 1 — Neural network for CHD classification
**CS-433 Machine Learning · EPFL**

This notebook implements, selects and evaluates a feed-forward neural network on the
`dataset/` split of the Project 1 classification task. The target `_MICHD` indicates
whether a respondent reports myocardial infarction or coronary heart disease.

**Libraries used:** the Python standard library, NumPy and matplotlib only — no pandas,
scikit-learn or deep-learning framework. The neural network, the preprocessing, the
metrics (ROC-AUC, average precision, precision/recall/F1) and the submission writing are
all implemented with NumPy in `implementations.py` and `nn_pipeline.py`.

### Scope
This is the code-only repository: it contains the neural-network notebook, the model and
preprocessing code, the six course methods, the grading tests and the submitted CSV. The
two-page report and the ablation scripts/outputs are kept locally and are not committed;
the notebook writes only local result artifacts under `results/`.

### Deliverables produced by this notebook
* `results/y_test.csv` — official competition submission (five-seed ensemble, reproduced by `run.py`).
* `results/y_test_notebook_single_seed.csv` — single-seed demonstration output (local).
* `results/tables/*.csv`, `results/figures/*.pdf` — local analysis artifacts (not committed).
* `results/nn_metrics_notebook_single_seed.json` — single-seed notebook metrics (local).

## 1. Data loading and exploratory overview

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np

# Make sure the project root (with nn_pipeline.py and the up-to-date
# implementations.py) is imported ahead of the stale copies in notebooks/.
ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dataset").is_dir() and (path / "implementations.py").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import nn_pipeline as nnp
from implementations import neural_network_predict, neural_network_train

RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
TABLES = RESULTS / "tables"
for _folder in (FIGURES, TABLES):
    _folder.mkdir(parents=True, exist_ok=True)

SEED = 42
raw = nnp.load_raw(ROOT / "dataset")
y = raw["y_train"]

print("Training features :", raw["x_train"].shape)
print("Test features     :", raw["x_test"].shape)
print("Training labels   :", y.shape, "| unique classes:", np.unique(y))

Training features : (328135, 321)
Test features     : (109379, 321)
Training labels   : (328135,) | unique classes: [-1.  1.]


The dataset is strongly **imbalanced**: only about 8.8% of the respondents are positive.
This is why accuracy is a poor metric here (predicting "no CHD" everywhere already reaches
~91% accuracy) and the analysis below focuses on the positive-class **F1 score**, the
threshold-free **ROC-AUC** and **average precision (AP)**.

In [2]:
counts = [int((y == -1).sum()), int((y == 1).sum())]
class_rows = [
    ["-1 (no CHD)", counts[0], counts[0] / len(y)],
    ["+1 (CHD)", counts[1], counts[1] / len(y)],
]
nnp.write_csv(TABLES / "class_distribution.csv", ["label", "count", "fraction"], class_rows)
nnp.plot_class_distribution(y, FIGURES / "fig_class_distribution.pdf")
print(nnp.format_table(["label", "count", "fraction"], class_rows))

label        count   fraction
-----------  ------  --------
-1 (no CHD)  299160  0.9117  
+1 (CHD)     28975   0.0883  


## 2. Train / validation / holdout split

We use a three-way **stratified** split so that the model selection and the final
evaluation never touch the same rows:

1. **Holdout (20%)** — an internal test set, used only once at the very end for the
   reported F1 / ROC-AUC / AP and the confusion matrix.
2. The remaining 80% (**dev**) is split again into a **train** set (64% of all rows) and a
   **validation** set (16%) used for model selection and threshold tuning.

Stratification preserves the 8.8% positive rate in every subset.

In [3]:
# (1) 20% internal test set, never used for training or threshold selection.
dev_idx, holdout_idx = nnp.stratified_split(y, val_fraction=0.20, seed=SEED)
# (2) Split the remaining 80% ("dev") into 80/20 train/validation.
train_local, val_local = nnp.stratified_split(
    y[dev_idx], val_fraction=0.20, seed=SEED + 1
)
train_idx, val_idx = dev_idx[train_local], dev_idx[val_local]

split_rows = [
    ["train", len(train_idx), float((y[train_idx] == 1).mean())],
    ["validation", len(val_idx), float((y[val_idx] == 1).mean())],
    ["holdout (internal test)", len(holdout_idx), float((y[holdout_idx] == 1).mean())],
    ["total labeled", len(y), float((y == 1).mean())],
]
nnp.write_csv(TABLES / "split_summary.csv", ["subset", "rows", "positive_fraction"], split_rows)
print(nnp.format_table(["subset", "rows", "positive_fraction"], split_rows))

subset                   rows    positive_fraction
-----------------------  ------  -----------------
train                    210006  0.0883           
validation               52502   0.0883           
holdout (internal test)  65627   0.0883           
total labeled            328135  0.0883           


## 3. Preprocessing

About **45% of the feature values are missing**. Because the features are a mix of
ordinal survey answers and continuous variables, we

* **mean-impute** the missing entries with the observed column mean,
* **drop** all-missing and constant columns (they carry no signal),
* **append a binary indicator** for every column that has missing values, so the network
  can also use *which* answers were skipped (informative missingness),
* **standardise** every kept column.

With $\mu_j$ the observed mean and $\sigma_j$ the observed standard deviation of column
$j$,

$$\tilde x_{ij}=\begin{cases}x_{ij},&x_{ij}\ \text{observed}\\ \mu_j,&\text{otherwise}\end{cases},
\qquad z_{ij}=\frac{\tilde x_{ij}-\mu_j}{\sigma_j}.$$

All statistics are estimated on the **training rows only** and then applied unchanged to
the validation, holdout and test rows, so there is no leakage.

In [4]:
Z_train, info = nnp.fit_design_matrix(raw["x_train"][train_idx], add_indicators=True)
Z_val = nnp.transform_design_matrix(raw["x_train"][val_idx], info)
Z_holdout = nnp.transform_design_matrix(raw["x_train"][holdout_idx], info)
Z_test = nnp.transform_design_matrix(raw["x_test"], info)

y_train01 = (y[train_idx] == 1).astype(float)
y_val01 = (y[val_idx] == 1).astype(float)
y_holdout01 = (y[holdout_idx] == 1).astype(float)

prep_rows = [
    ["raw features", int(info["keep"].size)],
    ["kept base features", info["n_base"]],
    ["missing indicators", info["n_indicator"]],
    ["final width", int(info["n_base"] + info["n_indicator"])],
    ["dropped columns", info["dropped"]],
    ["missing fraction (train)", round(info["missing_fraction"], 4)],
]
nnp.write_csv(TABLES / "preprocessing_summary.csv", ["quantity", "value"], prep_rows)
nnp.plot_missingness(
    raw["x_train"], raw["feature_names"], FIGURES / "fig_missingness.pdf"
)
print(
    "Z_train", Z_train.shape, "| Z_val", Z_val.shape,
    "| Z_holdout", Z_holdout.shape, "| Z_test", Z_test.shape,
)
print(nnp.format_table(["quantity", "value"], prep_rows))

Z_train (210006, 551) | Z_val (52502, 551) | Z_holdout (65627, 551) | Z_test (109379, 551)
quantity                  value 
------------------------  ------
raw features              321   
kept base features        315   
missing indicators        236   
final width               551   
dropped columns           6     
missing fraction (train)  0.4478


## 4. Neural network model and training objective

The network is a fully-connected **multi-layer perceptron**: one or more ReLU layers
followed by a single sigmoid output that returns the positive-class probability,

$$h^{(1)}=\mathrm{ReLU}(W_1 x + b_1),\quad\dots,\quad p=\sigma(W_L h^{(L-1)} + b_L).$$

Training minimises the **class-weighted binary cross-entropy** with L2 weight decay

$$L=-\frac{1}{N}\sum_i a_i\big[y_i\log p_i+(1-y_i)\log(1-p_i)\big]+\lambda\lVert W\rVert_2^2,
\qquad a_i=\begin{cases}\text{pos\_weight},&y_i=1\\1,&\text{otherwise}\end{cases},$$

optimised with **Adam** over mini-batches. The `pos_weight` term counteracts the 8.8%
positive rate, and L2 keeps the network from memorising the sparse one-hot-like survey
columns. The snippet below trains one such network and records the train/validation loss.

In [5]:
t0 = time.time()
params_demo, history = neural_network_train(
    y_train01,
    Z_train,
    hidden_dims=(32,),
    max_iters=8000,
    gamma=1e-3,
    batch_size=1024,
    seed=SEED,
    l2=1e-3,
    pos_weight=3.0,
    optimizer="adam",
    record_every=500,
    valid=(y_val01, Z_val),
)
print(
    f"trained in {time.time() - t0:.1f}s | "
    f"final train loss {history['train_loss'][-1]:.4f} | "
    f"final val loss {history['val_loss'][-1]:.4f}"
)
nnp.plot_learning_curve(history, FIGURES / "fig_learning_curve.pdf")
history_rows = [
    [iteration, train_loss, val_loss]
    for iteration, train_loss, val_loss in zip(
        history["iterations"], history["train_loss"], history["val_loss"]
    )
]
nnp.write_csv(
    TABLES / "training_history.csv",
    ["iteration", "train_loss", "val_loss"],
    history_rows,
)
print(nnp.format_table(["iteration", "train_loss", "val_loss"], history_rows))

trained in 8.8s | final train loss 0.4138 | final val loss 0.4391
iteration  train_loss  val_loss
---------  ----------  --------
500        0.4394      0.4444  
1000       0.4325      0.4407  
1500       0.4287      0.4389  
2000       0.4267      0.4374  
2500       0.4262      0.4386  
3000       0.4246      0.4396  
3500       0.4230      0.4392  
4000       0.4266      0.4448  
4500       0.4223      0.4397  
5000       0.4200      0.4394  
5500       0.4195      0.4418  
6000       0.4164      0.4372  
6500       0.4163      0.4395  
7000       0.4147      0.4396  
7500       0.4148      0.4398  
8000       0.4138      0.4391  


## 5. Model selection

We compare a small grid over **capacity** (one shallow layer vs. deeper networks) and the
two main regularisation levers, **positive-class weight** (`pos_weight`) and **L2**
(`l2`). Selection uses the **validation F1 at the F1-optimal threshold**, with ROC-AUC and
AP reported alongside. All runs share the same optimiser settings, seed and preprocessing.

In [6]:
configs = [
    ("hidden=32, pw=1, l2=0", dict(hidden_dims=(32,), pos_weight=1.0, l2=0.0)),
    ("hidden=32, pw=3, l2=0", dict(hidden_dims=(32,), pos_weight=3.0, l2=0.0)),
    ("hidden=32, pw=3, l2=1e-3", dict(hidden_dims=(32,), pos_weight=3.0, l2=1e-3)),
    ("hidden=16, pw=3, l2=1e-3", dict(hidden_dims=(16,), pos_weight=3.0, l2=1e-3)),
    ("hidden=16, pw=5, l2=1e-3", dict(hidden_dims=(16,), pos_weight=5.0, l2=1e-3)),
    ("hidden=64, pw=3, l2=1e-3", dict(hidden_dims=(64,), pos_weight=3.0, l2=1e-3)),
    ("hidden=64-32, pw=3, l2=1e-3", dict(hidden_dims=(64, 32), pos_weight=3.0, l2=1e-3)),
    ("hidden=128-64, pw=3, l2=1e-3", dict(hidden_dims=(128, 64), pos_weight=3.0, l2=1e-3)),
]

selection_rows = []
best = None
for name, cfg in configs:
    t0 = time.time()
    params, hist = neural_network_train(
        y_train01,
        Z_train,
        gamma=1e-3,
        batch_size=1024,
        seed=SEED,
        optimizer="adam",
        max_iters=8000,
        record_every=4000,
        valid=(y_val01, Z_val),
        **cfg,
    )
    val_probs = neural_network_predict(Z_val, params)
    threshold, f1_best = nnp.best_f1_threshold(y_val01, val_probs)
    metrics_05 = nnp.classification_metrics(y_val01, val_probs, 0.5)
    metrics_best = nnp.classification_metrics(y_val01, val_probs, threshold)

    selection_rows.append(
        {
            "config": name,
            "train_loss": hist["train_loss"][-1],
            "val_loss": hist["val_loss"][-1],
            "f1@0.5": metrics_05["f1"],
            "roc_auc": metrics_05["roc_auc"],
            "average_precision": metrics_05["average_precision"],
            "f1_best": f1_best,
            "threshold": threshold,
            "precision": metrics_best["precision"],
            "recall": metrics_best["recall"],
            "seconds": time.time() - t0,
        }
    )
    if best is None or f1_best > best["f1_best"]:
        best = dict(selection_rows[-1], config_dict=cfg, val_probs=val_probs)

sel_headers = [
    "config", "train_loss", "val_loss", "f1@0.5", "roc_auc",
    "average_precision", "f1_best", "threshold", "precision", "recall", "seconds",
]
sel_rows = [[row[key] for key in sel_headers] for row in selection_rows]
sel_rows.sort(key=lambda row: row[sel_headers.index("f1_best")], reverse=True)
nnp.write_csv(TABLES / "model_selection.csv", sel_headers, sel_rows)
print("Selected configuration:", best["config"], "| validation F1 =", round(best["f1_best"], 4))
print(nnp.format_table(sel_headers, sel_rows))

Selected configuration: hidden=16, pw=3, l2=1e-3 | validation F1 = 0.4337
config                        train_loss  val_loss  f1@0.5  roc_auc  average_precision  f1_best  threshold  precision  recall  seconds
----------------------------  ----------  --------  ------  -------  -----------------  -------  ---------  ---------  ------  -------
hidden=16, pw=3, l2=1e-3      0.4214      0.4371    0.4216  0.8610   0.3954             0.4337   0.4134     0.3558     0.5554  6.8375 
hidden=16, pw=5, l2=1e-3      0.5548      0.5864    0.4236  0.8588   0.3888             0.4303   0.5603     0.3678     0.5186  6.8407 
hidden=32, pw=3, l2=1e-3      0.4138      0.4391    0.4202  0.8596   0.3861             0.4301   0.4184     0.3510     0.5552  8.2754 
hidden=64, pw=3, l2=1e-3      0.4115      0.4408    0.4148  0.8581   0.3818             0.4258   0.4448     0.3552     0.5313  11.7152
hidden=64-32, pw=3, l2=1e-3   0.4136      0.4558    0.4018  0.8547   0.3787             0.4237   0.3881     0.3514  

## 6. Final evaluation on the internal holdout

The selected configuration is **refit on the full dev set** (train + validation, with the
preprocessing statistics re-estimated on those rows) and evaluated **once** on the 20%
holdout. We report the metrics both at the default threshold 0.5 and at the threshold that
maximised the validation F1, together with the confusion matrix and the ROC / PR curves.

In [7]:
best_cfg = best["config_dict"]
chosen_threshold = float(best["threshold"])

# Refit preprocessing on the full dev set and train the selected network there.
Z_dev, info_dev = nnp.fit_design_matrix(raw["x_train"][dev_idx], add_indicators=True)
Z_holdout_dev = nnp.transform_design_matrix(raw["x_train"][holdout_idx], info_dev)
y_dev01 = (y[dev_idx] == 1).astype(float)

t0 = time.time()
params_final, hist_final = neural_network_train(
    y_dev01,
    Z_dev,
    gamma=1e-3,
    batch_size=1024,
    seed=SEED,
    optimizer="adam",
    max_iters=8000,
    record_every=4000,
    **best_cfg,
)
holdout_probs = neural_network_predict(Z_holdout_dev, params_final)
print(f"final model trained on dev in {time.time() - t0:.1f}s")

metrics_at_half = nnp.classification_metrics(y_holdout01, holdout_probs, 0.5)
metrics_at_thr = nnp.classification_metrics(y_holdout01, holdout_probs, chosen_threshold)

eval_headers = [
    "setting", "threshold", "precision", "recall", "f1",
    "accuracy", "roc_auc", "average_precision", "tn", "fp", "fn", "tp",
]
eval_rows = [
    [f"default threshold 0.5", *[metrics_at_half[k] for k in eval_headers[1:]]],
    [
        f"validation-tuned threshold {chosen_threshold:.3f}",
        *[metrics_at_thr[k] for k in eval_headers[1:]],
    ],
]
nnp.write_csv(TABLES / "holdout_metrics.csv", eval_headers, eval_rows)
print(nnp.format_table(eval_headers, eval_rows))

final model trained on dev in 7.3s
setting                           threshold  precision  recall  f1      accuracy  roc_auc  average_precision  tn     fp    fn    tp  
--------------------------------  ---------  ---------  ------  ------  --------  -------  -----------------  -----  ----  ----  ----
default threshold 0.5             0.5000     0.3796     0.4937  0.4292  0.8841    0.8629   0.3964             55157  4675  2934  2861
validation-tuned threshold 0.413  0.4134     0.3330     0.6126  0.4315  0.8575    0.8629   0.3964             52722  7110  2245  3550


In [8]:
nnp.plot_confusion_matrix(metrics_at_thr, FIGURES / "fig_confusion_matrix.pdf")
nnp.plot_roc_curve(y_holdout01, holdout_probs, FIGURES / "fig_roc_curve.pdf")
nnp.plot_precision_recall_curve(
    y_holdout01, holdout_probs, FIGURES / "fig_precision_recall.pdf"
)
nnp.plot_threshold_curve(
    y_holdout01, holdout_probs, FIGURES / "fig_threshold_curve.pdf", chosen_threshold
)
nnp.plot_probability_distribution(
    y_holdout01, holdout_probs, FIGURES / "fig_probability_distribution.pdf"
)

confusion_headers = ["", "pred -1", "pred +1"]
confusion_rows = [
    ["true -1", metrics_at_thr["tn"], metrics_at_thr["fp"]],
    ["true +1", metrics_at_thr["fn"], metrics_at_thr["tp"]],
]
nnp.write_csv(TABLES / "confusion_matrix.csv", confusion_headers, confusion_rows)
print(nnp.format_table(confusion_headers, confusion_rows))

         pred -1  pred +1
-------  -------  -------
true -1  52722    7110   
true +1  2245     3550   


## 7. Test predictions and submission file

The official competition submission is produced by **`run.py`**, which trains the
selected five-seed probability ensemble on all labelled rows, reuses the
validation-calibrated threshold and writes `results/y_test.csv`. It reproduces the
submitted file bit-for-bit (competition F1 = 0.444, accuracy = 0.882).

The cell below illustrates the same procedure with a **single seed** (for speed) and
writes `results/y_test_notebook_single_seed.csv`, so it does **not** overwrite the
official submission. The notebook metrics are written to
`results/nn_metrics_notebook_single_seed.json`.


In [9]:
Z_all, info_all = nnp.fit_design_matrix(raw["x_train"], add_indicators=True)
Z_test_all = nnp.transform_design_matrix(raw["x_test"], info_all)
y_all01 = (y == 1).astype(float)

t0 = time.time()
params_submit, hist_submit = neural_network_train(
    y_all01,
    Z_all,
    gamma=1e-3,
    batch_size=1024,
    seed=SEED,
    optimizer="adam",
    max_iters=8000,
    record_every=4000,
    **best_cfg,
)
test_probs = neural_network_predict(Z_test_all, params_submit)
submission_path = nnp.save_submission(
    raw["test_ids"], test_probs, RESULTS / "y_test_notebook_single_seed.csv", threshold=chosen_threshold
)
print(f"submission written to {submission_path} in {time.time() - t0:.1f}s")

predicted_positive = int((test_probs >= chosen_threshold).sum())
print(
    "rows:", len(test_probs),
    "| predicted +1:", predicted_positive,
    f"({predicted_positive / len(test_probs):.2%})",
)
with open(submission_path) as handle:
    preview = [next(handle) for _ in range(6)]
print("".join(preview))

submission written to /Users/christinebau/Documents/epfl/Machine_Learning/project1/results/y_test.csv in 7.9s
rows: 109379 | predicted +1: 14848 (13.57%)
Id,Prediction
328135,-1
328136,-1
328137,-1
328138,-1
328139,-1



In [10]:
metrics = {
    "selected_config": best["config"],
    "note": "single-seed notebook output; the official submission is run.py (5-seed ensemble)",
    "chosen_threshold": chosen_threshold,
    "validation": {
        "f1": best["f1_best"],
        "roc_auc": best["roc_auc"],
        "average_precision": best["average_precision"],
    },
    "holdout_default_threshold": metrics_at_half,
    "holdout_tuned_threshold": metrics_at_thr,
    "test": {
        "rows": int(len(test_probs)),
        "predicted_positive": int((test_probs >= chosen_threshold).sum()),
        "predicted_positive_fraction": float((test_probs >= chosen_threshold).mean()),
        "mean_probability": float(test_probs.mean()),
    },
}
with open(RESULTS / "nn_metrics_notebook_single_seed.json", "w") as handle:
    json.dump(metrics, handle, indent=2)
print(json.dumps({k: v for k, v in metrics.items() if k != "test"}, indent=2))
print("test summary:", json.dumps(metrics["test"], indent=2))

{
  "selected_config": "hidden=16, pw=3, l2=1e-3",
  "chosen_threshold": 0.4133720539300956,
  "validation": {
    "f1": 0.43372073437763176,
    "roc_auc": 0.8610192417017496,
    "average_precision": 0.395360739371252
  },
  "holdout_default_threshold": {
    "threshold": 0.5,
    "precision": 0.3796443736730361,
    "recall": 0.49370146678170834,
    "f1": 0.42922511439501915,
    "accuracy": 0.884056866838344,
    "roc_auc": 0.8629028146800688,
    "average_precision": 0.3964308220733995,
    "tn": 55157,
    "fp": 4675,
    "fn": 2934,
    "tp": 2861
  },
  "holdout_tuned_threshold": {
    "threshold": 0.4133720539300956,
    "precision": 0.3330206378986867,
    "recall": 0.6125970664365833,
    "f1": 0.4314797933758736,
    "accuracy": 0.8574519633687354,
    "roc_auc": 0.8629028146800688,
    "average_precision": 0.3964308220733995,
    "tn": 52722,
    "fp": 7110,
    "fn": 2245,
    "tp": 3550
  }
}
test summary: {
  "rows": 109379,
  "predicted_positive": 14848,
  "predicted_

## 8. Analysis

**Signal and complexity.** Validation F1, ROC-AUC and AP are essentially flat once the
hidden layer has 16–32 units; wider/deeper networks are slightly worse in the controlled
five-seed ablation (64 units: −0.003 F1; 64–32: −0.008 F1). A small network with L2 is
therefore cheaper and better, which suggests a decision boundary close to (piecewise)
linear.

**Class imbalance and thresholding.** At 8.8% positives, a threshold of 0.5 under-predicts.
The ablation shows that `pos_weight = 3` mainly shifts the operating point: F1 at 0.5 rises
from 0.202 to 0.425, but after tuning the threshold both variants have essentially the same tuned F1
(0.434 vs 0.431) and the unweighted model even has slightly higher AP (0.393 vs 0.391).
Threshold calibration, not the weighted loss, is what handles the imbalance. The threshold
is tuned on validation and applied once to the holdout.

**Missingness is informative but not dominant.** Removing the 236 per-column indicators
costs 0.006 F1 and 0.013 AP; a single missing-count feature does not recover the loss. This
refines the initial hypothesis: mean imputation already preserves much of the marginal
signal.

**Comparison with linear baselines.** Ridge (F1 0.422, AUC 0.850 on validation) is the
strongest convex model; the linear network with the same loss and optimiser matches it in F1
and is slightly better in AUC. The 16-unit MLP adds 0.011 F1 / 0.007 AUC, and five-seed
ensembling adds another 0.007 F1. On the internal holdout the ensemble reaches F1 0.4350,
ROC-AUC 0.8653 and AP 0.4037, versus 0.4233 / 0.8507 / 0.3956 for ridge. The collaborator's
SVM uses a smaller provisional split, so it is not directly comparable.

**Caveats and next steps.** One stratified split; five seeds capture initialisation but not
split variance. The network is a plain MLP; target-aware feature engineering or
gradient-boosted trees remain open. The controlled ablation, the same-split course-method
comparison and the holdout bootstrap are summarised in the separate two-page report; the
scripts and raw outputs are kept locally and are not part of this code-only repository.

---
**Reproducibility note.** The numbers printed above use one seed for speed. The competition
submission (`run.py`, `results/y_test.csv`) uses the five-seed ensemble and achieved
F1 = 0.444 and accuracy = 0.882 on the competition site; it is reproduced bit-for-bit. The ablation, course-method comparison and
holdout bootstrap are separate artifacts kept outside this code-only repository; their
summary is in the report.